In [2]:
import pandas as pd
import numpy as np
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Trỏ đường dẫn để gọi file trong thư mục src
sys.path.append(os.path.abspath('../src'))
from dataPreprocessing import load_and_preprocess_raw

RAW_DIR = '../dataset/raw/'

# 1. Load dữ liệu thông qua hàm đóng gói (Lấy mẫu 10000 dòng để phân tích nhanh)
print("Đang tải dữ liệu...")
df = load_and_preprocess_raw(RAW_DIR, nrows=10000)

print("\nĐang tính toán phân loại nhu cầu. Quá trình này có thể mất 1-2 phút...")

# Lấy mẫu 100 sản phẩm để phân tích
sample_items = df['item_id'].unique()[:100]
df_run = df[df['item_id'].isin(sample_items)]

def classify_demand(group):
    """Hàm tính ADI, CV2 và phân loại cho từng sản phẩm"""
    demand = group['demand'].values
    if demand.sum() == 0:
        return pd.Series({'ADI': np.nan, 'CV2': np.nan, 'Category': 'No Sales'})
        
    non_zero_indices = np.where(demand > 0)[0]
    first_sale_idx = non_zero_indices[0]
    active_demand = demand[first_sale_idx:]
    
    days_with_sales = len(active_demand[active_demand > 0])
    adi = len(active_demand) / days_with_sales if days_with_sales > 0 else np.nan
        
    mean_demand = np.mean(active_demand)
    cv2 = (np.std(active_demand) / mean_demand) ** 2 if mean_demand > 0 else 0
        
    if adi <= 1.32 and cv2 <= 0.49:
        category = 'Smooth (Đều đặn)'
    elif adi <= 1.32 and cv2 > 0.49:
        category = 'Erratic (Thất thường)'
    elif adi > 1.32 and cv2 <= 0.49:
        category = 'Intermittent (Ngắt quãng)'
    else:
        category = 'Lumpy (Cục bộ)'
        
    return pd.Series({'ADI': round(adi, 2), 'CV2': round(cv2, 2), 'Category': category})

# 2. Áp dụng hàm phân loại
classification_df = df_run.groupby(['item_id', 'store_id']).apply(classify_demand).reset_index()
classification_df = classification_df[classification_df['Category'] != 'No Sales']

print("\n✅ Hoàn tất! Thống kê số lượng các nhóm nhu cầu:")
print(classification_df['Category'].value_counts())

print("\nBảng phân loại chi tiết (5 dòng đầu):")
display(classification_df.head())

Đang tải dữ liệu...
1. Đang tải và ép cân dữ liệu Calendar...
---> Dung lượng ban đầu: 0.21 MB
---> Dung lượng sau khi tối ưu: 0.19 MB

2. Đang tải dữ liệu Sales...

3. Đang Melt (kéo giãn) dữ liệu Sales...

4. Đang Merge Sales với Calendar...

5. Ép cân lần cuối cho DataFrame tổng...
---> Dung lượng ban đầu: 1568.97 MB
---> Dung lượng sau khi tối ưu: 620.78 MB

Đang tính toán phân loại nhu cầu. Quá trình này có thể mất 1-2 phút...

✅ Hoàn tất! Thống kê số lượng các nhóm nhu cầu:
Category
Lumpy (Cục bộ)           385
Erratic (Thất thường)     14
Smooth (Đều đặn)           1
Name: count, dtype: int64

Bảng phân loại chi tiết (5 dòng đầu):


,item_id,store_id,ADI,CV2,Category
0,FOODS_1_001,CA_1,NaN,NaN,NaN
1,FOODS_1_001,CA_2,NaN,NaN,NaN
2,FOODS_1_001,CA_3,NaN,NaN,NaN
3,FOODS_1_001,CA_4,NaN,NaN,NaN
4,FOODS_1_002,CA_1,NaN,NaN,NaN
